# 02 — Silver conform

Reads the bronze Delta tables and writes conformed silver tables to `lh_silver`.

What this notebook does:
- Casts types, trims strings and deduplicates IDs.
- Normalises blank cause IDs to `Unknown`.
- Deduplicates station-level `target_oil`.
- Builds `silver_production_day`, `silver_deferment_event`, `silver_station_day`,
  `silver_deferment_episode`, the reference dimensions and `silver_data_quality`.
- Idempotent: each run overwrites the silver snapshot.


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
BRONZE_ID = "0cde2eb7-937b-49c6-a4ed-b95bd66dc49e"   # lh_bronze
BRONZE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"


def read_bronze(name: str):
    return spark.read.format("delta").load(f"{BRONZE}/Tables/bronze_{name}")


def write_silver(name: str, df) -> None:
    table = f"silver_{name}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    print(f"{table}: {df.count():,} rows")


def as_long(name: str):
    return F.trim(F.col(name).cast("string")).cast("long")


def as_double(name: str):
    return F.trim(F.col(name).cast("string")).cast("double")


def as_date(name: str):
    return F.to_date(F.trim(F.col(name).cast("string")))


stations_all = (
    read_bronze("dim_station").select(
        as_long("flowstation_id").alias("flowstation_id"),
        F.col("flowstation").alias("station_name"),
        as_long("asset_id").alias("asset_id"),
        F.col("asset").alias("asset"),
    )
)
stations_join = stations_all.select("flowstation_id", "station_name", "asset")


In [ ]:
raw_prod = read_bronze("production_fact")
w_prod = Window.partitionBy("production_id").orderBy(F.col("_ingested_at_utc").desc_nulls_last())
prod = (
    raw_prod
    .withColumn("_rn", F.row_number().over(w_prod))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        as_long("production_id").alias("production_id"),
        as_long("asset_id").alias("asset_id"),
        as_long("flowstation_id").alias("flowstation_id"),
        as_long("reservoir_id").alias("reservoir_id"),
        as_long("date_id").alias("date_id"),
        F.trim(F.col("productionstring")).alias("productionstring"),
        F.trim(F.col("status")).alias("status"),
        as_double("gas").alias("gas"),
        as_double("gross").alias("gross"),
        as_double("oil").alias("oil"),
        as_double("target_oil").alias("target_oil"),
        as_double("uptimeproduction").alias("uptime_hrs"),
        as_double("water").alias("water"),
        as_double("water_rf").alias("water_rf"),
        as_double("thp").alias("thp"),
        as_double("bean").alias("bean"),
        as_date("production_date").alias("production_date"),
        F.col("_batch_id"),
        F.col("_ingested_at_utc"),
        F.col("_source_file"),
    )
)

# In the source, target_oil repeats at station level on every string row.
w_station_day = Window.partitionBy("flowstation_id", "production_date")
prod = prod.withColumn("station_target_oil", F.max("target_oil").over(w_station_day))
prod = prod.join(stations_join, "flowstation_id", "left")

write_silver("production_day", prod)


In [ ]:
raw_defer = read_bronze("deferment_fact")
w_def = Window.partitionBy("deferment_id").orderBy(F.col("_ingested_at_utc").desc_nulls_last())
defer = (
    raw_defer
    .withColumn("_rn", F.row_number().over(w_def))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        as_long("deferment_id").alias("deferment_id"),
        as_long("asset_id").alias("asset_id"),
        as_long("flowstation_id").alias("flowstation_id"),
        as_long("reservoir_id").alias("reservoir_id"),
        as_long("date_id").alias("date_id"),
        as_long("category_id").alias("category_id"),
        as_long("subcategory_id").alias("subcategory_id"),
        F.trim(F.col("productionstring")).alias("productionstring"),
        F.trim(F.col("status")).alias("status"),
        as_double("gross").alias("deferred_gross"),
        as_double("water").alias("deferred_water"),
        as_double("oil").alias("deferred_oil"),
        as_double("gas").alias("deferred_gas"),
        as_date("date").alias("event_date"),
        F.col("_batch_id"),
        F.col("_ingested_at_utc"),
        F.col("_source_file"),
    )
)

cat = read_bronze("deferment_category_dim").select(
    as_long("category_id").alias("category_id"),
    F.col("category").alias("cause_group"),
)
sub = read_bronze("deferment_subcategory_dim").select(
    as_long("subcategory_id").alias("subcategory_id"),
    F.col("sub_category").alias("cause_code"),
)

defer = (
    defer
    .join(cat, "category_id", "left")
    .join(sub, "subcategory_id", "left")
    .withColumn("cause_group", F.coalesce(F.col("cause_group"), F.lit("Unknown")))
    .withColumn("cause_code", F.coalesce(F.col("cause_code"), F.lit("Unknown")))
    .withColumn(
        "is_coded",
        F.col("subcategory_id").isNotNull()
        & (~F.lower(F.col("cause_code")).isin("mismatch", "unknown")),
    )
    .join(stations_join, "flowstation_id", "left")
    .withColumnRenamed("flowstation_id", "station_id")
)

write_silver("deferment_event", defer)


In [ ]:
prod_agg = (
    prod.groupBy("flowstation_id", "production_date")
    .agg(
        F.sum("oil").alias("actual_oil"),
        F.max("station_target_oil").alias("target_oil"),
        F.sum("gas").alias("actual_gas"),
        F.sum("water").alias("actual_water"),
        F.sum("gross").alias("actual_gross"),
        F.avg("uptime_hrs").alias("uptime_avg"),
        F.avg("thp").alias("thp_avg"),
        F.avg("bean").alias("bean_avg"),
        F.avg("water_rf").alias("water_rf_avg"),
        F.countDistinct("productionstring").alias("wells_total"),
        F.countDistinct(F.when(F.col("oil") > 0, F.col("productionstring"))).alias("wells_producing"),
    )
    .withColumnRenamed("production_date", "date")
)

defer_agg = (
    defer.groupBy("station_id", "event_date")
    .agg(
        F.sum("deferred_oil").alias("deferred_oil"),
        F.count("*").alias("event_count"),
        F.sum(F.col("is_coded").cast("int")).alias("coded_event_count"),
    )
    .withColumnRenamed("station_id", "flowstation_id")
    .withColumnRenamed("event_date", "date")
)

sd = prod_agg.join(defer_agg, ["flowstation_id", "date"], "full_outer")

for c in [
    "actual_oil", "deferred_oil", "actual_gas", "actual_water", "actual_gross",
    "uptime_avg", "thp_avg", "bean_avg", "water_rf_avg",
    "wells_total", "wells_producing", "event_count", "coded_event_count",
]:
    sd = sd.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

sd = (
    sd
    .withColumn("potential_oil", F.col("actual_oil") + F.col("deferred_oil"))
    .withColumn(
        "deferment_pct",
        F.when(F.col("potential_oil") > 0, F.col("deferred_oil") / F.col("potential_oil")),
    )
    .withColumn(
        "target_gap",
        F.when(F.col("target_oil").isNotNull(), F.col("target_oil") - F.col("actual_oil")),
    )
    .withColumn(
        "target_gap_pct",
        F.when(F.col("target_oil") > 0, F.col("target_gap") / F.col("target_oil")),
    )
    .join(stations_join, "flowstation_id", "left")
    .withColumnRenamed("flowstation_id", "station_id")
)

write_silver("station_day", sd)


In [ ]:
w_ep = Window.partitionBy("station_id", "productionstring", "cause_code").orderBy("event_date")
ev = defer.select(
    "station_id", "productionstring", "cause_code", "cause_group",
    "event_date", "deferred_oil", "is_coded",
)
ev = (
    ev
    .withColumn("prev_date", F.lag("event_date").over(w_ep))
    .withColumn(
        "is_new",
        F.when(
            F.col("prev_date").isNull() | (F.datediff("event_date", "prev_date") > 1),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "episode_seq",
        F.sum("is_new").over(w_ep.rowsBetween(Window.unboundedPreceding, Window.currentRow)),
    )
)

ep = (
    ev.groupBy("station_id", "productionstring", "cause_code", "cause_group", "episode_seq")
    .agg(
        F.min("event_date").alias("episode_start"),
        F.max("event_date").alias("episode_end"),
        F.count("*").alias("event_count"),
        F.sum("deferred_oil").alias("deferred_oil"),
        F.max(F.col("is_coded").cast("int")).alias("has_coded_event"),
    )
    .withColumn("duration_days", F.datediff("episode_end", "episode_start") + 1)
    .withColumn("has_coded_event", F.col("has_coded_event").cast("boolean"))
    .withColumn(
        "episode_id",
        F.concat_ws("|", "station_id", "productionstring", "cause_code", "episode_seq"),
    )
    .drop("episode_seq")
)

write_silver("deferment_episode", ep)


In [ ]:
write_silver("dim_date", read_bronze("dim_date").select(
    as_long("date_id").alias("date_id"),
    as_date("date").alias("date"),
    as_long("year").alias("year"),
    as_long("month").alias("month"),
    as_long("day").alias("day"),
    F.col("weekday"),
    F.col("is_weekend").cast("boolean").alias("is_weekend"),
))

write_silver("dim_station", stations_all)

write_silver("dim_well", read_bronze("dim_well").select(
    F.col("productionstring"),
    as_long("flowstation_id").alias("flowstation_id"),
    as_long("asset_id").alias("asset_id"),
))

write_silver("dim_cause_group", read_bronze("deferment_category_dim").select(
    as_long("category_id").alias("cause_group_id"),
    F.col("category").alias("cause_group"),
))

write_silver("dim_cause_code", read_bronze("deferment_subcategory_dim").select(
    as_long("subcategory_id").alias("cause_code_id"),
    F.col("sub_category").alias("cause_code"),
).withColumn("is_usable", ~F.lower(F.col("cause_code")).isin("mismatch", "unknown")))

write_silver("oil_prices", read_bronze("oil_prices").select(
    as_date("date").alias("date"),
    as_double("oil_price_usd").alias("oil_price_usd"),
))

write_silver("owner_map", read_bronze("owner_map").select(
    as_long("flowstation_id").alias("station_id"),
    F.col("flowstation").alias("station_name"),
    F.col("asset"),
    F.col("owner_name"),
    F.col("owner_upn"),
))


In [ ]:
prod_rows = prod.count()
defer_rows = defer.count()
coded_rows = defer.filter(F.col("is_coded")).count()
unknown_rows = defer.filter(F.col("cause_code") == "Unknown").count()
sd_rows = sd.count()
batch = prod.agg(F.max("_batch_id")).first()[0]

metrics = [
    ("batch_id", batch),
    ("production_rows", prod_rows),
    ("deferment_events", defer_rows),
    ("coded_events", coded_rows),
    ("coded_event_pct", round(coded_rows / defer_rows, 4) if defer_rows else 0.0),
    ("unknown_cause_events", unknown_rows),
    ("unknown_cause_pct", round(unknown_rows / defer_rows, 4) if defer_rows else 0.0),
    ("station_days", sd_rows),
    ("stations", sd.select("station_id").distinct().count()),
    ("wells", prod.select("productionstring").distinct().count()),
    ("min_date", str(sd.agg(F.min("date")).first()[0])),
    ("max_date", str(sd.agg(F.max("date")).first()[0])),
    ("production_duplicate_ids", raw_prod.count() - raw_prod.select("production_id").distinct().count()),
    ("deferment_duplicate_ids", raw_defer.count() - raw_defer.select("deferment_id").distinct().count()),
    ("thp_missing_pct", round(prod.filter(F.col("thp").isNull()).count() / prod_rows, 4) if prod_rows else 0.0),
    ("bean_missing_pct", round(prod.filter(F.col("bean").isNull()).count() / prod_rows, 4) if prod_rows else 0.0),
]

dq = spark.createDataFrame([(k, str(v)) for k, v in metrics], ["metric", "value"])
write_silver("data_quality", dq)


In [ ]:
display(
    spark.sql("SHOW TABLES")
    .filter(F.col("tableName").startswith("silver_"))
    .orderBy("tableName")
)
display(
    sd.orderBy(F.col("date").desc(), F.col("deferment_pct").desc()).limit(15)
)
